# Frozen Seven-Day Holdout Results

This notebook reviews the published results of a queue-aware BTC/USDT market-making experiment. Twelve configurations were trained on 42 chronological days and frozen before evaluation on September 13-19, 2026.

The purpose is to inspect the result, not to refit or tune against the holdout.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'results' / 'published').exists():
    ROOT = ROOT.parent
PUBLISHED = ROOT / 'results' / 'published'

summary = pd.read_csv(PUBLISHED / 'model_summary.csv')
daily = pd.read_csv(PUBLISHED / 'best_strategy_daily.csv', parse_dates=['date'])
all_daily = pd.read_csv(PUBLISHED / 'daily_pnl.csv', parse_dates=['date'])

pd.set_option('display.max_columns', None)
display(summary.sort_values('total_pnl_usd', ascending=False).reset_index(drop=True))

## Comparison across all 12 configurations

Only the 600-second unified-long configuration is positive. This model directly predicts the full net outcome and includes 28 short- and long-context features.

In [ ]:
ordered = summary.sort_values('total_pnl_usd')
labels = [
    f"{int(row.horizon_seconds)}s | {row.objective.replace('_', ' ').title()}"
    for row in ordered.itertuples()
]
values = ordered['total_pnl_usd'].to_numpy()
colors = np.where(values > 0, '#2a9d8f', '#607d8b')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels, values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set(title='Frozen Seven-Day Holdout', xlabel='Total net PnL (USD)')
plt.tight_layout()
plt.show()

## Best configuration by day

The T=600 unified-long strategy earned $43.87 across the seven days and was positive on six days. The result is dominated by a $105.24 loss on September 15 and therefore remains statistically fragile.

In [ ]:
display(daily[['date', 'pnl_usd', 'fills', 'submitted_orders', 'max_abs_inventory_btc']])

colors = np.where(daily['pnl_usd'] >= 0, '#2a9d8f', '#d1495b')
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(daily['date'].dt.strftime('%b %d'), daily['pnl_usd'], color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set(title='T=600 Unified Long: Daily Net PnL', ylabel='Net PnL (USD)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Interpretation

The seven-day result supports a limited conclusion: under this simulator, direct long-context net-outcome prediction selected better quotes than the other tested objectives during this specific week. It does **not** establish a deployable strategy.

A later, untouched extension through September 30 made the same frozen strategy negative. The main failure mode was inventory exposure on strong trend days. See `docs/FOLLOW_UP.md` for the documented follow-up.